In [1]:
import joblib
import pandas as pd
import numpy as np
import pandas as pd

# Show all columns
pd.set_option('display.max_columns', None)

# Show complete text in each column
pd.set_option('display.max_colwidth', None)

# Prevent wrapping by increasing display width
pd.set_option('display.width', None)



 Load processed_data & trained model

In [2]:
df=pd.read_csv('../data/processed_telco_churn.csv')
lr= joblib.load("../models/customer_churn_model.pkl")
X=df.drop('Churn',axis=1)

Predict churn

In [3]:
df["predicted_churn"] = lr.predict(X)


 Predict churn probability

In [4]:
df["Churn_Probability"] = lr.predict_proba(X)[:, 1]

Calculate CLV, Priority Score

In [5]:

df["CLV"]=df["MonthlyCharges"]*df["tenure"]
# Priority Score (0–100)
df["Priority_Score"] = (
    df["Churn_Probability"] * 0.7 +
    (df["CLV"] / df["CLV"].max()) * 0.3
) * 100

df["Priority_Score"] = df["Priority_Score"].round(2)


Assign Priority Level to each customer

In [6]:
def priority_level(score):
    if score >= 80:
        return "Critical"
    elif score >= 60:
        return "High"
    elif score >= 40:
        return "Medium"
    else:
        return "Low"

df["Priority_Level"] = df["Priority_Score"].apply(priority_level)


Create Smart Retention Recommendation

In [15]:
def smart_retention(row):

    priority = row["Priority_Level"]
    clv = row["CLV"]
    tenure = row["tenure"]
    monthly = row["MonthlyCharges"]
    churn = row["Churn_Probability"]

    # =========================
    # CRITICAL
    # =========================

    if priority == "Critical":

        if churn >= 0.95 and clv >= 5000:
            return "Immediate Manager Call + 30% Discount + Dedicated Relationship Manager"

        elif tenure < 6:
            return "Free 3-Month Subscription + Personal Onboarding"

        elif row["Contract"] == "Month-to-month":
            return "Offer Free Upgrade to One-Year Contract"

        elif monthly >= 100:
            return "25% Discount + Premium Support"

        elif (
            row["OnlineSecurity"] == "No"
            and row["TechSupport"] == "No"
        ):
            return "Free Online Security & Tech Support for 6 Months"

        elif row["OnlineSecurity"] == "No":
            return "Free Online Security for 6 Months"

        elif row["TechSupport"] == "No":
            return "Free Technical Support for 6 Months"

        elif row["OnlineBackup"] == "No":
            return "Free Cloud Backup for 3 Months"

        elif row["InternetService"] == "Fiber optic":
            return "Free Speed Upgrade + Premium Support"

        elif row["PaymentMethod"] == "Electronic check":
            return "Switch to Auto-Pay and Get ₹500 Cashback"

        else:
            return "20% Loyalty Discount"

    # =========================
    # HIGH
    # =========================

    elif priority == "High":

        if clv >= 5000:
            return "Premium Support + Free Service Upgrade"

        elif tenure < 12:
            return "Welcome Back Offer + Free Installation"

        elif monthly >= 100:
            return "25% Discount"

        elif row["Contract"] == "Month-to-month":
            return "Offer One-Year Contract with Discount"

        elif row["InternetService"] == "Fiber optic":
            return "Free Speed Upgrade"

        elif row["OnlineSecurity"] == "No":
            return "Free Online Security Trial"

        elif row["TechSupport"] == "No":
            return "Free Technical Support Trial"

        elif row["OnlineBackup"] == "No":
            return "Free Cloud Backup Trial"

        elif row["PaymentMethod"] == "Electronic check":
            return "Auto-Pay Cashback Offer"

        elif row["SeniorCitizen"] == 1:
            return "Senior Citizen Special Discount"

        else:
            return "15% Discount + Loyalty Points"

    # =========================
    # MEDIUM
    # =========================

    elif priority == "Medium":

        if tenure < 12:
            return "Welcome Offer + Personalized Email"

        elif monthly >= 90:
            return "10% Discount on Monthly Bill"

        elif row["Contract"] == "Month-to-month":
            return "Recommend Annual Contract"

        elif row["OnlineSecurity"] == "No":
            return "Free Online Security Trial"

        elif row["TechSupport"] == "No":
            return "Free Technical Support Trial"

        elif row["OnlineBackup"] == "No":
            return "Free Cloud Backup Trial"

        elif row["PaymentMethod"] == "Electronic check":
            return "Recommend Auto-Pay"

        elif row["SeniorCitizen"] == 1:
            return "Senior Citizen Discount Plan"

        else:
            return "Personalized Marketing Email"

    # =========================
    # LOW
    # =========================

    else:

        if clv >= 5000:
            return "VIP Loyalty Rewards"

        elif tenure >= 60:
            return "Anniversary Reward Coupon"

        elif row["Contract"] == "Two year":
            return "Early Renewal Bonus"

        elif row["InternetService"] == "Fiber optic":
            return "Free Streaming Service Trial"

        elif row["PaymentMethod"] == "Credit card (automatic)":
            return "Cashback Reward"

        else:
            return "Regular Promotional Email"

df["Retention_Action"]=df.apply(smart_retention,axis=1)

# Recommedation Reason

In [16]:
def recommendation_reason(row):

    priority = row["Priority_Level"]
    clv = row["CLV"]
    tenure = row["tenure"]
    monthly = row["MonthlyCharges"]
    churn = row["Churn_Probability"]

    # =========================
    # CRITICAL
    # =========================

    if priority == "Critical":

        if churn >= 0.95 and clv >= 5000:
            return "Customer has very high churn probability and very high lifetime value."

        elif tenure < 6:
            return "Customer is new and has a high risk of churn."

        elif row["Contract"] == "Month-to-month":
            return "Customer is on a month-to-month contract, which has higher churn risk."

        elif monthly >= 100:
            return "Customer has high monthly charges."

        elif (
            row["OnlineSecurity"] == "No"
            and row["TechSupport"] == "No"
        ):
            return "Customer is not subscribed to Online Security and Tech Support."

        elif row["OnlineSecurity"] == "No":
            return "Customer is not using Online Security."

        elif row["TechSupport"] == "No":
            return "Customer is not using Technical Support."

        elif row["OnlineBackup"] == "No":
            return "Customer is not using Online Backup."

        elif row["InternetService"] == "Fiber optic":
            return "Customer uses Fiber optic internet, which showed higher churn behaviour in the dataset."

        elif row["PaymentMethod"] == "Electronic check":
            return "Customer uses Electronic Check, which showed relatively higher churn."

        else:
            return "Customer has been identified as a critical retention case."

    # =========================
    # HIGH
    # =========================

    elif priority == "High":

        if clv >= 5000:
            return "Customer has high lifetime value and should be retained."

        elif tenure < 12:
            return "Customer is still in the early stage of the relationship."

        elif monthly >= 100:
            return "Customer has high monthly charges."

        elif row["Contract"] == "Month-to-month":
            return "Customer is using a month-to-month contract."

        elif row["InternetService"] == "Fiber optic":
            return "Customer uses Fiber optic internet."

        elif row["OnlineSecurity"] == "No":
            return "Customer has not subscribed to Online Security."

        elif row["TechSupport"] == "No":
            return "Customer has not subscribed to Technical Support."

        elif row["OnlineBackup"] == "No":
            return "Customer has not subscribed to Online Backup."

        elif row["PaymentMethod"] == "Electronic check":
            return "Customer uses Electronic Check payment."

        elif row["SeniorCitizen"] == 1:
            return "Customer is a senior citizen and may benefit from targeted offers."

        else:
            return "Customer has been identified as a high-priority retention case."

    # =========================
    # MEDIUM
    # =========================

    elif priority == "Medium":

        if tenure < 12:
            return "Customer is relatively new."

        elif monthly >= 90:
            return "Customer has relatively high monthly charges."

        elif row["Contract"] == "Month-to-month":
            return "Customer is using a month-to-month contract."

        elif row["OnlineSecurity"] == "No":
            return "Customer is not using Online Security."

        elif row["TechSupport"] == "No":
            return "Customer is not using Technical Support."

        elif row["OnlineBackup"] == "No":
            return "Customer is not using Online Backup."

        elif row["PaymentMethod"] == "Electronic check":
            return "Customer uses Electronic Check payment."

        elif row["SeniorCitizen"] == 1:
            return "Customer may benefit from senior-focused offers."

        else:
            return "Customer has moderate churn risk."

    # =========================
    # LOW
    # =========================

    else:

        if clv >= 5000:
            return "Customer is highly valuable and currently has low churn priority."

        elif tenure >= 60:
            return "Customer has been with the company for a long time."

        elif row["Contract"] == "Two year":
            return "Customer has a long-term contract, which generally indicates stronger retention."

        elif row["InternetService"] == "Fiber optic":
            return "Customer uses Fiber internet but currently has low churn priority."

        elif row["PaymentMethod"] == "Credit card (automatic)":
            return "Customer uses automatic payment and currently has low churn priority."

        else:
            return "Customer currently has low churn risk."

In [17]:
df["Recommendation_Reason"] = df.apply(recommendation_reason, axis=1)

In [18]:
retention_report = df[
    [
        "Churn_Probability",
        "CLV",
        "Priority_Score",
        "Priority_Level",
        "MonthlyCharges",
        "tenure",
        "Retention_Action",
        "Recommendation_Reason"
    ]
].sort_values(
    by=["Priority_Score", "Churn_Probability"],
    ascending=[False, False]
)

display(retention_report.head(20))



,Churn_Probability,CLV,Priority_Score,Priority_Level,MonthlyCharges,tenure,Retention_Action,Recommendation_Reason
6380,0.709636,6782.75,73.47,High,104.35,65,Premium Support + Free Service Upgrade,Customer has high lifetime value and should be retained.
2879,0.688114,7206.50,73.45,High,102.95,70,Premium Support + Free Service Upgrade,Customer has high lifetime value and should be retained.
4181,0.731369,6283.70,73.24,High,101.35,62,Premium Support + Free Service Upgrade,Customer has high lifetime value and should be retained.
2075,0.762092,5658.80,73.20,High,101.05,56,Premium Support + Free Service Upgrade,Customer has high lifetime value and should be retained.
4074,0.753723,5724.60,72.85,High,98.70,58,Premium Support + Free Service Upgrade,Customer has high lifetime value and should be retained.
3330,0.790397,4987.80,72.83,High,97.80,51,Offer One-Year Contract with Discount,Customer is using a month-to-month contract.
3200,0.756805,5632.20,72.74,High,104.30,54,Premium Support + Free Service Upgrade,Customer has high lifetime value and should be retained.
7023,0.709726,6520.50,72.56,High,103.50,63,Premium Support + Free Service Upgrade,Customer has high lifetime value and should be retained.
6032,0.677245,7160.40,72.53,High,105.30,68,Premium Support + Free Service Upgrade,Customer has high lifetime value and should be retained.
2609,0.712607,6438.55,72.47,High,105.55,61,Premium Support + Free Service Upgrade,Customer has high lifetime value and should be retained.


In [19]:
df.to_csv("customer_retention_result.csv", index=False)

print("Retention results saved successfully!")

Retention results saved successfully!
